# BraTS2023 — YOLO26-S-Seg train / val / test

Baseline supervised nhị phân, dùng T1c / T2-FLAIR / T2w làm 3 kênh ảnh 2D.
Chia theo bệnh nhân 70/15/15, seed 29; ưu tiên 47 bản sửa MEN; loại validation không có nhãn.
Đây chưa phải thí nghiệm chuyển miền MedRT-SFSeg hoặc điểm challenge BraTS chính thức.

Chọn runtime GPU. Đặt repo và `dataset/BraTS2023` trên Drive, hoặc sửa các đường dẫn dưới đây.
Toàn bộ dữ liệu raw khoảng 40 GiB. Chuyển đổi và train đầy đủ có thể vượt thời lượng một phiên Colab.
Kết quả/checkpoint lưu trên Drive; ảnh huấn luyện được chuẩn bị trên ổ local của runtime.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_PROJECT = Path('/content/drive/MyDrive/MedRT-SFOD')
WORK = Path('/content/MedRT-SFOD')
RAW_SOURCE = DRIVE_PROJECT / 'dataset/BraTS2023'
OUTPUT_PROJECT = DRIVE_PROJECT / 'runs/seg/brats2023'
DOMAINS = ['GLI', 'MEN', 'PED']  # Có thể chọn một nhóm cho lần chạy đầu.
RUN_NAME = 'yolo26s_wt'
EPOCHS = 100
BATCH = 8
IMAGE_SIZE = 256
RESUME = False  # True nếu cùng RUN_NAME đã có weights/last.pt và protocol.json.


In [ ]:
import os, shutil, subprocess, sys
assert (DRIVE_PROJECT / 'scripts/YOLO26/medseg/prepare_brats2023.py').is_file(), 'Cần copy phiên bản repo đã có script BraTS lên Drive.'
assert RAW_SOURCE.is_dir(), RAW_SOURCE
WORK.mkdir(parents=True, exist_ok=True)
for folder in ('ultralytics', 'scripts'):
    shutil.copytree(DRIVE_PROJECT / folder, WORK / folder, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
for name in ('pyproject.toml', 'README.md', 'requirements-brats.txt'):
    shutil.copy2(DRIVE_PROJECT / name, WORK / name)
os.chdir(WORK)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-r', 'requirements-brats.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Hãy bật GPU runtime trước khi train đầy đủ.'
print(torch.cuda.get_device_name(0))


## Chuẩn bị dữ liệu

Đọc trực tiếp NIfTI từ Drive và ghi ảnh đã chuyển đổi vào ổ local, tránh thêm một bản sao raw 40 GiB.
Script chuẩn hóa từng volume, giữ mọi lát cắt, kiểm tra hình học và xuất mask gốc để đánh giá.
Không chọn lát cắt dựa trên nhãn. Bước này có thể chạy lâu do tốc độ đọc Drive.
Nếu chuyển đổi bị ngắt, chọn thư mục đích mới hoặc chủ động xóa bản chuyển đổi chưa hoàn tất rồi chạy lại.


In [ ]:
import json
PREPARED = WORK / ('dataset/BraTS2023-YOLO26-' + '-'.join(DOMAINS))
DATA = PREPARED / 'dataset_seg.yaml'
if not DATA.is_file():
    subprocess.run([sys.executable, 'scripts/YOLO26/medseg/prepare_brats2023.py',
                    '--src', str(RAW_SOURCE), '--dst', str(PREPARED), '--domains', *DOMAINS], check=True)
manifest = json.loads((PREPARED / 'manifest.json').read_text())
assert manifest['prepared'] and not manifest['smoke_subset']
assert sorted({c['domain'] for c in manifest['cases']}) == sorted(DOMAINS)
print(json.dumps(manifest['audit']['split_counts'], indent=2))
print('Split fingerprint:', manifest['split_sha256'])
# Giữ bản kiểm kê cùng kết quả trên Drive.
OUTPUT_PROJECT.mkdir(parents=True, exist_ok=True)
shutil.copy2(PREPARED / 'manifest.json', OUTPUT_PROJECT / (RUN_NAME + '_manifest.json'))


## Train, rồi val và test

Mô hình pretrained YOLO26-S-Seg được tải khi cần. `best.pt` chọn bằng fitness trên val.
Test chỉ được dùng sau khi chọn checkpoint; confidence Dice/IoU cố định 0,25.
Nếu training bị ngắt, đặt `RESUME=True` với cùng đường dẫn và cấu hình. Script dùng `last.pt`.
Khi chỉ cần đánh giá một checkpoint đã train xong, dùng hướng dẫn `--eval-only` trong docs/BraTS2023.md.


In [ ]:
cmd = [sys.executable, 'scripts/YOLO26/medseg/run_brats2023.py',
       '--data', str(DATA), '--weights', 'yolo26s-seg.pt',
       '--epochs', str(EPOCHS), '--imgsz', str(IMAGE_SIZE),
       '--batch', str(BATCH), '--workers', '2', '--device', '0',
       '--project', str(OUTPUT_PROJECT), '--name', RUN_NAME]
if RESUME:
    cmd.append('--resume')
subprocess.run(cmd, check=True)


In [ ]:
report = json.loads((OUTPUT_PROJECT / RUN_NAME / 'evaluation.json').read_text())
assert not report['smoke_subset'], 'Kết quả smoke không dùng kết luận chất lượng.'
for split in ('val', 'test'):
    print('\n' + split.upper())
    for domain, stats in report[split]['by_domain'].items():
        print(domain, 'patients=', stats['patients'],
              'Dice=', round(stats['patient_macro']['dice'], 4),
              'IoU=', round(stats['patient_macro']['iou'], 4),
              'Dice 95% CI=', stats['dice_patient_bootstrap_95ci'])
    print('YOLO mAP:', report[split]['yolo_metrics'])
print('Artifacts:', OUTPUT_PROJECT / RUN_NAME)


Cần xem kết quả từng nhóm GLI/MEN/PED, độ chênh val–test và các ca dự đoán kém trước khi kết luận.
Dice/IoU được tính theo thể tích từng ca từ mask gốc, rồi trung bình theo bệnh nhân.
Đây là baseline một lớp; không so sánh trực tiếp với điểm ET/TC/WT lesion-wise của challenge.
Để đánh giá đóng góp MedRT-SFSeg, chạy thí nghiệm source/target riêng và so sánh với source-only,
supervised target và từng ablation trên cùng holdout.
